<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 4 — Dashboard Power BI & Storytelling
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Ce notebook est un **guide d'implémentation Power BI**. Chaque section correspond à une étape de construction du dashboard.  
> Les cellules Python préparent et exportent les données. Les blocs `MÉTHODE` expliquent les choix de design et DAX.

| | |
|---|---|
| **Prérequis** | NB2 et NB3 complétés — tous les CSV disponibles dans `SAVE_PATH` |
| **Outils** | Power BI Desktop + Python (préparation données) |
| **Durée estimée** | 4h à 5h |

> **Livrable :** Dashboard 5 pages couvrant la vue executive, les corridors, les transporteurs,
> le CSAT et les alertes ML — avec 12 mesures DAX et un bandeau rouge conditionnel.

---
## 0. Préparation des données pour Power BI

> **MÉTHODE — Quels fichiers charger dans Power BI ?**
>
> Power BI chargera **7 sources de données** depuis `SAVE_PATH` :
> - `logitrack_analytics.csv` → table de faits principale
> - `logitrack_corridors.csv` → agrégats corridors (NB3)
> - `logitrack_transporteurs_perf.csv` → matrice transporteurs (NB3)
> - `logitrack_mensuel.csv` → tendance mensuelle (NB3)
> - `logitrack_cout_retard.csv` → coût financier (NB3)
> - `logitrack_entrepots_perf.csv` → entrepôts goulots (NB3)
> - `logitrack_risque_scores.csv` → alertes ML (NB5)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Dossier : {SAVE_PATH}')

# Vérification des fichiers nécessaires
fichiers_requis = [
    'logitrack_analytics.csv',
    'logitrack_corridors.csv',
    'logitrack_transporteurs_perf.csv',
    'logitrack_mensuel.csv',
    'logitrack_cout_retard.csv',
    'logitrack_entrepots_perf.csv',
    'logitrack_risque_scores.csv',
]
print('\nVérification des fichiers :')
for f in fichiers_requis:
    path = os.path.join(SAVE_PATH, f)
    status = '✅' if os.path.exists(path) else '❌ MANQUANT — exécuter le NB correspondant'
    print(f'  {status} {f}')

---
## Étape 1 — Table Calendrier

### MÉTHODE
La table Calendrier est obligatoire pour les analyses temporelles dans Power BI. Elle permet :
- Le filtrage par période (année, trimestre, mois)
- L'utilisation des fonctions `DATEADD`, `SAMEPERIODLASTYEAR`, `TOTALYTD`
- La saisonnalité et les comparaisons an vs an

On la crée en DAX directement dans Power BI avec `CALENDARAUTO()` puis on y ajoute les colonnes calculées.

### Code DAX — Table Calendrier

Dans Power BI → **Modélisation → Nouvelle table** :

```dax
Calendrier =
VAR DateMin = MIN(livraisons[date_creation])
VAR DateMax = MAX(livraisons[date_creation])
RETURN
ADDCOLUMNS(
    CALENDAR(DateMin, DateMax),
    "Année",         YEAR([Date]),
    "Mois",          MONTH([Date]),
    "Mois Nom",      FORMAT([Date], "MMM YYYY"),
    "Mois Nom Court",FORMAT([Date], "MMM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date]),
    "Jour Semaine",  WEEKDAY([Date], 2),
    "Jour Nom",      FORMAT([Date], "dddd"),
    "Est Weekend",   IF(WEEKDAY([Date], 2) >= 6, 1, 0),
    "Tri Mois",      YEAR([Date]) * 100 + MONTH([Date])
)
```

> **Puis créer la relation :** `Calendrier[Date]` → `livraisons[date_creation]` (Many to One)

---
## Étape 2 — Modèle de données

### MÉTHODE — Schéma en étoile
Power BI performe mieux avec un **schéma en étoile** : une table de faits centrale (`livraisons`) et des tables de dimensions autour. Les tables d'agrégats NB3 sont chargées comme tables indépendantes — elles ne se joignent pas à `livraisons` mais sont utilisées directement pour les visuels agrégés.

### Relations à créer dans Power BI

```
Calendrier
  [Date] ──────────────── livraisons [date_creation]     (1:N)

-- Tables de référence --
transporteurs_perf
  [transporteur] ──────── livraisons [transporteur_id]   (1:N, inactif)

corridors
  [corridor] ──────────── livraisons [corridor calculé]  (table indépendante)

alertes_ml
  (table indépendante — pas de relation directe)
```

> **Note :** Les tables `logitrack_corridors.csv`, `logitrack_mensuel.csv`,
> `logitrack_cout_retard.csv` et `logitrack_entrepots_perf.csv` sont des tables
> pré-agrégées depuis NB3. Elles s'utilisent directement sans relation
> — évite les problèmes de contexte de filtre DAX sur des agrégats complexes.

---
## Étape 3 — Les 12 mesures DAX

### MÉTHODE
Créer toutes les mesures dans une **table de mesures dédiée** (table vide nommée `_Mesures`) :
Power BI → **Entrer des données** → créer une table vide → y placer toutes les mesures.
Cela facilite la maintenance et la navigation dans le panneau Champs.

### Mesures de base (groupe : KPIs Globaux)

```dax
-- 1. Total Livraisons
Total Livraisons =
COUNTROWS(livraisons)

-- 2. Taux SLA Breach
Taux Breach % =
DIVIDE(
    CALCULATE(COUNTROWS(livraisons), livraisons[sla_breach] = 1),
    [Total Livraisons]
) * 100

-- 3. CSAT Moyen
CSAT Moyen =
CALCULATE(
    AVERAGEX(
        FILTER(livraisons, NOT ISBLANK(livraisons[csat])
               && livraisons[csat] >= 1 && livraisons[csat] <= 5),
        livraisons[csat]
    )
)

-- 4. Total Pénalités
Total Pénalités FCFA =
SUM(livraisons[penalite_fcfa])

-- 5. Nb Escalades
Nb Escalades =
CALCULATE(COUNTROWS(livraisons), livraisons[escalade] = 1)
```

### Mesures de comparaison (groupe : Évolution)

```dax
-- 6. Taux Breach Mois Précédent
Taux Breach Mois Précédent % =
CALCULATE(
    [Taux Breach %],
    DATEADD(Calendrier[Date], -1, MONTH)
)

-- 7. Variation Breach vs Mois Précédent
Variation Breach % =
VAR Current  = [Taux Breach %]
VAR Previous = [Taux Breach Mois Précédent %]
RETURN
IF(NOT ISBLANK(Previous), Current - Previous, BLANK())

-- 8. Taux Breach YTD
Taux Breach YTD % =
DIVIDE(
    CALCULATE(
        COUNTROWS(livraisons),
        livraisons[sla_breach] = 1,
        DATESYTD(Calendrier[Date])
    ),
    CALCULATE(
        COUNTROWS(livraisons),
        DATESYTD(Calendrier[Date])
    )
) * 100
```

### Mesures ML Alertes (groupe : Alertes ML)

```dax
-- 9. Nb Alertes Critiques
Nb Alertes Critiques =
CALCULATE(
    COUNTROWS(logitrack_risque_scores),
    logitrack_risque_scores[niveau_urgence] = "Critique"
)

-- 10. Nb Total Alertes
Nb Alertes Total =
CALCULATE(
    COUNTROWS(logitrack_risque_scores),
    logitrack_risque_scores[alerte_retard] = 1
)

-- 11. Bandeau Alerte Texte
Bandeau Alerte =
VAR NbCrit = [Nb Alertes Critiques]
RETURN
IF(
    NbCrit > 0,
    "⚠️ " & NbCrit & " livraisons en risque critique — intervention requise",
    "✅ Aucune alerte critique en cours"
)

-- 12. Quadrant Agent (pour la matrice scatter)
Quadrant Transporteur =
VAR BreachMed = CALCULATE(MEDIAN(logitrack_transporteurs_perf[taux_breach_pct]))
VAR CsatMed   = CALCULATE(MEDIAN(logitrack_transporteurs_perf[csat_moyen]))
VAR Breach    = SELECTEDVALUE(logitrack_transporteurs_perf[taux_breach_pct])
VAR Csat      = SELECTEDVALUE(logitrack_transporteurs_perf[csat_moyen])
RETURN
SWITCH(TRUE(),
    Breach <= BreachMed && Csat >= CsatMed, "Top Performer",
    Breach <= BreachMed && Csat <  CsatMed, "Fiable Peu Apprécié",
    Breach >  BreachMed && Csat >= CsatMed, "Apprécié Peu Fiable",
    "À Surveiller"
)
```

---
## Étape 4 — Architecture du dashboard 5 pages

### Page 1 — Vue Executive
*Première page vue par la direction — synthèse en un coup d'œil*

**Bandeau haut (fond #E24B4A si alertes critiques, #1D9E75 sinon)**
- Carte texte : mesure `Bandeau Alerte`
- Formatage conditionnel sur la couleur de fond

**KPIs (4 cartes en ligne)**

| Carte | Mesure | Format |
|---|---|---|
| Total Livraisons | `Total Livraisons` | Nombre entier |
| Taux Breach | `Taux Breach %` | `0.0%` — rouge si > 35% |
| CSAT Moyen | `CSAT Moyen` | `0.00` — vert si > 4.0 |
| Total Pénalités | `Total Pénalités FCFA` | Milliers FCFA |

**Visuels principaux**
- **Graphique en courbe** : `Total Livraisons` + `Taux Breach %` par `Calendrier[Mois Nom]`
  - Axe Y gauche : volume (barres bleues)
  - Axe Y droit : taux breach (ligne rouge)
- **Carte choroplèthe** : taux breach par `pays_destination` (si carte Bing disponible)
  - Sinon : graphique en barres horizontales pays × taux breach
- **Tableau** : Top 5 corridors par `Total Pénalités FCFA` décroissant

**Filtres (panneau gauche)**
- Segment : `Calendrier[Année]`
- Segment : `Calendrier[Trimestre]`
- Liste déroulante : `livraisons[priorite]`

### Page 2 — Analyse des Corridors
*Répondre à : quelles routes concentrent les retards ?*

**Graphique principal — Barres horizontales empilées**
- Source : `logitrack_corridors.csv`
- Axe Y : `corridor`
- Valeur 1 : `taux_breach_pct` (rouge)
- Valeur 2 : `100 - taux_breach_pct` (vert)
- Ligne de référence : moyenne globale (trait pointillé jaune)
- Trier par `taux_breach_pct` décroissant

**Scatter plot — Distance × Taux breach**
- Axe X : `distance_km`
- Axe Y : `taux_breach_pct`
- Taille bulle : `nb_livraisons`
- Couleur : `risque_douanier` (Faible=vert, Moyen=orange, Élevé=rouge)
- Infobulle : `corridor`, `delai_moy_retard_j`, `penalites_fcfa`

**Tableau de détail**
- Colonnes : `corridor`, `risque_douanier`, `nb_livraisons`,
  `taux_breach_pct`, `delai_moy_retard_j`, `penalites_fcfa`, `rang_risque`
- Formatage conditionnel sur `taux_breach_pct` : gradient rouge

**Filtre :** Segment `risque_douanier` (Faible / Moyen / Élevé)

### Page 3 — Performance des Transporteurs
*Répondre à : quels partenaires sont fiables et à quel coût ?*

**Scatter plot — Matrice quadrant**
- Source : `logitrack_transporteurs_perf.csv`
- Axe X : `taux_breach_pct`
- Axe Y : `csat_moyen`
- Taille bulle : `nb_livraisons`
- Étiquette : `transporteur`
- Lignes de référence : médiane X + médiane Y (4 quadrants)
- Annotations quadrants : Top Performer / À Surveiller / etc.

**Graphique barres — Pénalités totales**
- Axe X : `transporteur`
- Valeur : `total_penalites`
- Couleur conditionnelle : rouge si `tier_fiabilite = 3`

**Tableau comparatif**
- Colonnes : `transporteur`, `mode_transport`, `nb_livraisons`,
  `taux_breach_pct`, `csat_moyen`, `duree_moy_j`, `cout_moy_fcfa`,
  `ratio_penalite_cout_pct`, `rang_fiabilite`
- Icônes conditionnelles sur `rang_fiabilite` : 🟢 Top 3 / 🟡 Milieu / 🔴 Bas

**KPI cards**
- Meilleur taux breach (MIN)
- Pire taux breach (MAX)
- Économie possible si remplacement des transporteurs tier 3 par tier 1

### Page 4 — CSAT & Qualité de Service
*Répondre à : la satisfaction client est-elle corrélée au taux de breach ?*

**Graphique courbe double axe**
- Source : `logitrack_mensuel.csv`
- Axe X : `mois`
- Axe Y gauche : `csat_moy` (ligne violette, axe 1-5)
- Axe Y droit : `taux_breach_pct` (ligne rouge, axe 0-100)
- Objectif : montrer la corrélation inverse CSAT ↓ quand breach ↑

**Histogramme distribution CSAT**
- Source : `livraisons`
- Axe X : `csat` (arrondi à l'entier)
- Valeur : `Total Livraisons`
- Couleur : gradient rouge (1) → vert (5)

**Tableau CSAT par corridor**
- Source : `logitrack_corridors.csv`
- Colonnes : `corridor`, `csat_moyen`, `taux_breach_pct`, `nb_escalades`
- Trier par `csat_moyen` croissant

**KPI cards**
- CSAT Moyen global
- % livraisons avec CSAT ≥ 4
- Nb escalades total

### Page 5 — Alertes ML & Prédictions
*Page opérationnelle — ouverte chaque matin par les dispatchers*

**Bandeau haut conditionnel (rouge si Nb Alertes Critiques > 0)**
- Mesure : `Bandeau Alerte`
- Police blanche, fond rouge (#E24B4A) ou vert (#1D9E75)

**3 KPI cards (alertes ML)**

| Carte | Valeur | Description |
|---|---|---|
| Alertes Critiques | `Nb Alertes Critiques` | Score > 0.75 |
| Alertes Élevées | `Nb Alertes Total - Critiques` | Score 0.55-0.75 |
| Score Risque Moyen | `AVERAGE(score_risque)` | Sur les alertes déclenchées |

**Tableau d'alertes prioritaires**
- Source : `logitrack_risque_scores.csv`
- Filtré sur : `alerte_retard = 1`
- Colonnes : `livraison_id`, `pays_origine`, `pays_destination`,
  `priorite`, `date_livraison_prevue`, `score_risque`, `niveau_urgence`
- Trier par `score_risque` décroissant
- Formatage conditionnel `niveau_urgence` : Critique=rouge, Élevé=orange, Modéré=jaune

**Graphique barres — Alertes par corridor**
- Axe X : `pays_origine || ' → ' || pays_destination`
- Valeur : `COUNTROWS` filtré `alerte_retard = 1`

**Graphique en anneau — Répartition par niveau d'urgence**
- Critique / Élevé / Modéré
- Couleurs : rouge / orange / jaune

> **Note :** Cette page utilise `logitrack_risque_scores.csv` produit par NB5. Sans NB5, elle reste vide. Prévoir un message de substitution : `IF(ISBLANK([Nb Alertes Total]), "Exécuter le NB5 pour activer les alertes ML", [Nb Alertes Total])`

---
## Étape 5 — Préparation complémentaire Python

### MÉTHODE
Certaines transformations sont plus simples en Python qu'en DAX. On prépare ici deux tables supplémentaires :
- `logitrack_corridor_label.csv` : libellés normalisés pour l'affichage dans Power BI
- `logitrack_kpi_summary.csv` : tableau de synthèse pour la page Vue Executive

In [ ]:
import pandas as pd
import os

# Charger la table analytique
clean_path = os.path.join(SAVE_PATH, 'logitrack_analytics.csv')
if not os.path.exists(clean_path):
    print('⚠️  logitrack_analytics.csv non trouvé — fichier de démo')
else:
    df = pd.read_csv(clean_path, parse_dates=['date_creation'])
    df = df[df['transporteur_actif'] == 1]

    # Table corridor enrichie pour Power BI
    df['corridor'] = df['pays_origine'] + ' → ' + df['pays_destination']
    corridor_pbi = (
        df.groupby(['corridor','pays_origine','pays_destination',
                    'risque_douanier','risque_douanier_num'])
        .agg(
            nb_livraisons   = ('livraison_id','count'),
            taux_breach_pct = ('sla_breach', lambda x: round(x.mean()*100, 1)),
            csat_moyen      = ('csat', lambda x: round(x.dropna().mean(), 2)),
            penalites_fcfa  = ('penalite_fcfa', 'sum'),
            delai_moy_j     = ('delai_jours', lambda x: round(x[x>0].mean(), 1))
        ).reset_index()
        .sort_values('taux_breach_pct', ascending=False)
        .reset_index(drop=True)
    )
    corridor_pbi['rang_risque'] = corridor_pbi['taux_breach_pct'].rank(
        ascending=False).astype(int)

    corridor_pbi.to_csv(f'{SAVE_PATH}logitrack_pbi_corridors.csv', index=False)
    print(f'✅ logitrack_pbi_corridors.csv : {len(corridor_pbi)} corridors')

In [ ]:
if os.path.exists(clean_path):
    # KPI Summary pour la Vue Executive
    kpi_mensuel = (
        df.assign(mois=df['date_creation'].dt.to_period('M').astype(str))
        .groupby('mois')
        .agg(
            nb_livraisons   = ('livraison_id','count'),
            nb_breach       = ('sla_breach','sum'),
            taux_breach_pct = ('sla_breach', lambda x: round(x.mean()*100, 1)),
            csat_moy        = ('csat', lambda x: round(x.dropna().mean(), 2)),
            penalites_fcfa  = ('penalite_fcfa','sum'),
            nb_escalades    = ('escalade','sum'),
        ).reset_index()
        .sort_values('mois')
    )
    kpi_mensuel['variation_breach'] = kpi_mensuel['taux_breach_pct'].diff().round(1)

    kpi_mensuel.to_csv(f'{SAVE_PATH}logitrack_pbi_mensuel.csv', index=False)
    print(f'✅ logitrack_pbi_mensuel.csv : {len(kpi_mensuel)} mois')
    print(kpi_mensuel[['mois','taux_breach_pct','csat_moy','nb_escalades']].to_string())

---
## Étape 6 — Synthèse exécutive & Recommandations

### MÉTHODE
La synthèse exécutive est le livrable final présenté à la direction. Elle doit répondre à une question simple : **si vous avez 5 minutes pour lire ce dashboard, quels sont les 3 points à retenir et les 2 actions immédiates ?**

### Synthèse exécutive — LogiTrack Supply Chain Analytics

---

**Contexte :** Analyse de 15 000 livraisons sur 24 mois (janv. 2022 — déc. 2023) couvrant 6 pays, 15 transporteurs et 12 entrepôts.

---

**3 constats majeurs**

**① Le taux de breach structurel (39%) ne s'améliore pas**
Sur 24 mois, le taux de breach oscille autour de 39% sans tendance baissière. Les pics de novembre-janvier (+5 à +8 points) coïncident avec les pics de volume — le réseau n'a pas de capacité buffer pour absorber la saisonnalité.

**② 3 transporteurs concentrent 60% des retards**
TRP12 (NordSud Express), TRP05 et TRP06 opèrent principalement sur les corridors sahéliens à risque douanier élevé. Leur tarif bas est trompeur : le coût total (tarif + pénalités + impact CSAT) les rend plus coûteux que TRP07 ou TRP03.

**③ Le retard commence à l'entrepôt, pas chez le transporteur**
Les entrepôts de Bamako et Ouagadougou ont un délai de préparation moyen 40% supérieur aux hubs d'Abidjan et Douala. Une partie significative du retard est créée avant même l'enlèvement par le transporteur.

---

**2 recommandations prioritaires**

**① Actions immédiates (0-3 mois)**
- Interdire les enlèvements du vendredi après 14h sur les corridors > 1 500 km
- Renforcer les équipes de dispatch à Bamako et Ouagadougou (+1 agent par shift)
- Suspendre TRP12 sur les corridors Mali et Burkina Faso — rediriger vers TRP07

**② Actions structurelles (3-12 mois)**
- Recalibrer les SLA contractuels des corridors à risque douanier élevé (+2 jours)
- Déployer le système d'alerte ML (NB5) en production — alertes quotidiennes aux dispatchers
- Lancer des appels d'offres sur les corridors sahéliens pour identifier un 4ème transporteur fiable

---

**ROI estimé du système d'alerte ML**

| Scénario | Retards évités | Économie estimée |
|---|---|---|
| Pessimiste (20% des alertes traitées) | ~1 170 retards/an | ~14 M FCFA/an |
| Réaliste (40% des alertes traitées) | ~2 340 retards/an | ~28 M FCFA/an |
| Optimiste (60% des alertes traitées) | ~3 510 retards/an | ~42 M FCFA/an |

---
## Bilan du Notebook 4

### Architecture du dashboard

| Page | Titre | Sources | Visuels clés |
|---|---|---|---|
| 1 | Vue Executive | `livraisons`, `mensuel` | Bandeau alerte, 4 KPI cards, courbe volume+breach, top corridors |
| 2 | Corridors | `pbi_corridors` | Barres empilées, scatter distance×breach, tableau détail |
| 3 | Transporteurs | `transporteurs_perf` | Scatter quadrant, barres pénalités, tableau comparatif |
| 4 | CSAT & Qualité | `mensuel`, `livraisons` | Courbe double axe, histogramme CSAT, tableau CSAT×corridor |
| 5 | Alertes ML | `risque_scores` | Bandeau rouge, tableau alertes prioritaires, anneau niveaux |

### 12 mesures DAX créées

| Groupe | Mesures |
|---|---|
| KPIs Globaux | `Total Livraisons`, `Taux Breach %`, `CSAT Moyen`, `Total Pénalités FCFA`, `Nb Escalades` |
| Évolution | `Taux Breach Mois Précédent %`, `Variation Breach %`, `Taux Breach YTD %` |
| Alertes ML | `Nb Alertes Critiques`, `Nb Alertes Total`, `Bandeau Alerte`, `Quadrant Transporteur` |

**Fichiers additionnels exportés :**
- `logitrack_pbi_corridors.csv`
- `logitrack_pbi_mensuel.csv`

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.